═══════════════════════════════════════════════════════════════════════════
📌 Notebook 7: Visualização em Mapas Geográficos
═══════════════════════════════════════════════════════════════════════════

Neste notebook vamos:
✅ Baixar shapefile do Brasil
✅ Criar mapa coroplético dos clusters
✅ Visualizar distribuição geográfica
✅ Criar mapas interativos com Plotly/Folium
✅ Identificar padrões regionais
✅ Gerar visualizações para apresentação

⏱️ Tempo estimado: 30-40 minutos

⚠️ NOTA: Este notebook é OPCIONAL e requer GeoPandas

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CÉLULA 1: Setup e verificação de dependências (COMPLETA)
# ═══════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

print("🗺️  VISUALIZAÇÃO EM MAPAS GEOGRÁFICOS")
print("="*80)
print()

# =========================
# CONFIGS DE PATH DO PROJETO
# =========================
# Ajuste se necessário:
# - notebook em /notebooks -> ".."
# - notebook na raiz do projeto -> "."
base_path = Path("..")

raw_dir = base_path / "data" / "raw"
processed_dir = base_path / "data" / "processed"
figures_dir = base_path / "outputs" / "figures"
results_dir = base_path / "outputs" / "results"

for d in [raw_dir, processed_dir, figures_dir, results_dir]:
    d.mkdir(parents=True, exist_ok=True)

# =========================
# CONFIGS DE PERFORMANCE
# =========================
SIMPLIFY_TOL = 0.02          # Plotly (0.01 mais detalhado | 0.05 bem leve)
FOLIUM_SIMPLIFY_TOL = 0.03   # Folium
PLOTLY_MAX_FEATURES = None   # None = todos; ex: 3000 para amostrar

print("📁 Diretórios do projeto:")
print(f"   base_path:     {base_path.resolve()}")
print(f"   raw_dir:       {raw_dir.resolve()}")
print(f"   processed_dir: {processed_dir.resolve()}")
print(f"   results_dir:   {results_dir.resolve()}")
print(f"   figures_dir:   {figures_dir.resolve()}")
print()

# =========================
# Verificar GeoPandas/Folium
# =========================
try:
    import geopandas as gpd
    import folium
    from folium import plugins
    GEOPANDAS_AVAILABLE = True
    print("✅ GeoPandas instalado!")
    print("✅ Folium instalado!")
except ImportError:
    GEOPANDAS_AVAILABLE = False
    print("⚠️  GeoPandas ou Folium NÃO está instalado.")
    print("   Instale com:")
    print("   pip install geopandas folium pyogrio")
print()

if not GEOPANDAS_AVAILABLE:
    print("❌ Este notebook requer GeoPandas e Folium para mapas com geometrias.")
    print("   Você ainda pode usar as visualizações SEM shapefile (CÉLULA 8).")
else:
    print("✅ Dependências OK! Vamos prosseguir.")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CÉLULA 2: Carregar dados (COMPLETA)
# ═══════════════════════════════════════════════════════════════════════════

print("="*80)
print("📂 CARREGANDO DADOS")
print("="*80)
print()

# Dataset com clusters
data_path = results_dir / "municipios_clustered_final.csv"
if not data_path.exists():
    raise FileNotFoundError(f"❌ Não encontrei o dataset: {data_path}")

df = pd.read_csv(data_path)
print(f"✅ Dataset carregado: {len(df):,} municípios")
print(f"📊 Colunas: {df.shape[1]}")
print()

# Nomes dos clusters
names_path = results_dir / "cluster_names.json"
if not names_path.exists():
    raise FileNotFoundError(f"❌ Não encontrei o arquivo: {names_path}")

with open(names_path, "r", encoding="utf-8") as f:
    cluster_names_data = json.load(f)

mapping = cluster_names_data.get("mapeamento")

# suporta dict { "0": "nome", ... } OU lista ["nome0","nome1",...]
if isinstance(mapping, dict):
    cluster_names = {int(k): v for k, v in mapping.items()}
elif isinstance(mapping, list):
    cluster_names = {i: v for i, v in enumerate(mapping)}
else:
    cluster_names = {}

print(f"✅ Nomes dos clusters carregados: {len(cluster_names)}")
print()

# Verificar código IBGE
if "codigo_ibge" not in df.columns:
    print("❌ ERRO: Coluna 'codigo_ibge' não encontrada no df!")
    print("   O shapefile precisa desta coluna para fazer o merge.")
else:
    nunique = df["codigo_ibge"].astype(str).str.strip().nunique()
    print(f"✅ Coluna codigo_ibge presente: {nunique:,} códigos únicos")

print()

# (opcional) garantir tipo/limpeza desde já
if "codigo_ibge" in df.columns:
    df["codigo_ibge"] = df["codigo_ibge"].astype(str).str.strip()

# garantir que cluster seja int quando existir
if "cluster" in df.columns:
    df["cluster"] = pd.to_numeric(df["cluster"], errors="coerce")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CÉLULA 3: Baixar/Carregar Shapefile do Brasil (COMPLETA e ROBUSTA)
# ═══════════════════════════════════════════════════════════════════════════

gdf = None  # garante variável definida

if GEOPANDAS_AVAILABLE:
    print("="*80)
    print("🗺️  CARREGANDO SHAPEFILE DOS MUNICÍPIOS")
    print("="*80)
    print()

    shp_dir = raw_dir / "municipios_brasil"
    shp_dir.mkdir(parents=True, exist_ok=True)

    shp_files = list(shp_dir.glob("*.shp"))

    if len(shp_files) == 0:
        print(f"⚠️  Nenhum .shp encontrado em: {shp_dir}")
        print()
        print("📌 Você já tem o código de download do IBGE funcionando.")
        print("   Extraia o ZIP exatamente em:")
        print(f"   {shp_dir}")
        print()
        print("✅ Quando estiver lá dentro, esta célula vai carregar automaticamente.")
    else:
        # pega o primeiro .shp encontrado (se tiver mais de um, você pode trocar manualmente)
        shapefile_path = shp_files[0]
        print(f"✅ Shapefile encontrado: {shapefile_path}")
        print("🔄 Carregando shapefile...")
        print()

        try:
            gdf = gpd.read_file(shapefile_path)
            print(f"✅ Shapefile carregado: {len(gdf):,} geometrias")
            print()
            print("📋 Colunas disponíveis:")
            print(gdf.columns.tolist())
            print()

            # possíveis colunas de código
            code_columns = [c for c in gdf.columns if ("cod" in c.lower()) or ("cd" in c.lower())]
            print(f"📌 Possíveis colunas de código: {code_columns}")

        except Exception as e:
            print(f"❌ Erro ao carregar shapefile: {e}")
            gdf = None


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CÉLULA 4: Preparar dados geográficos (merge) (COMPLETA e ROBUSTA)
# ═══════════════════════════════════════════════════════════════════════════

gdf_merged = None  # garante variável definida

if GEOPANDAS_AVAILABLE and (gdf is not None):
    print("\n" + "="*80)
    print("🔗 FAZENDO MERGE: DADOS + SHAPEFILE")
    print("="*80)
    print()

    if "codigo_ibge" not in df.columns:
        print("❌ Não dá para fazer merge: df não tem 'codigo_ibge'.")
    else:
        # detectar coluna de código no shapefile
        code_col = None
        for col in ["CD_MUN", "CD_GEOCODM", "codigo_ibge", "GEOCODIGO", "CD_GEOCMU"]:
            if col in gdf.columns:
                code_col = col
                break

        if code_col is None:
            print("❌ Não encontrei uma coluna de código IBGE no shapefile.")
            print("   Colunas disponíveis:", gdf.columns.tolist())
        else:
            print(f"✅ Coluna de código usada no shapefile: {code_col}")

            # padronizar chaves
            df_key = df["codigo_ibge"].astype(str).str.strip()
            gdf_key = gdf[code_col].astype(str).str.strip()

            df["codigo_ibge"] = df_key
            gdf = gdf.copy()
            gdf[code_col] = gdf_key

            print()
            print("✅ Chaves padronizadas:")
            print("   df[codigo_ibge] exemplos:", df["codigo_ibge"].head(3).tolist())
            print(f"   gdf[{code_col}] exemplos:", gdf[code_col].head(3).tolist())
            print()

            # reduzir df ao mínimo necessário (isso também deixa tudo mais leve)
            cols_needed = ["codigo_ibge", "cluster", "cluster_nome", "nome_municipio", "uf", "regiao"]
            cols_needed = [c for c in cols_needed if c in df.columns]

            df_merge = df[cols_needed].copy()

            # merge
            gdf_merged = gdf.merge(
                df_merge,
                left_on=code_col,
                right_on="codigo_ibge",
                how="left"
            )

            total = len(gdf_merged)
            matched = int(gdf_merged["cluster"].notna().sum())
            missing = int(gdf_merged["cluster"].isna().sum())
            pct_missing = (missing / total * 100) if total else 0

            print("📊 RESULTADO DO MERGE:")
            print(f"   Geometrias no shapefile: {total:,}")
            print(f"   Municípios com cluster (match): {matched:,}")
            print(f"   Sem match (NaN): {missing:,} ({pct_missing:.2f}%)")
            print()

            if missing > 0:
                print("⚠️ Exemplos sem match (primeiros 10):")
                show_cols = [c for c in [code_col, "NM_MUN", "SIGLA_UF"] if c in gdf_merged.columns]
                display(gdf_merged.loc[gdf_merged["cluster"].isna(), show_cols].head(10))
                print()

            # filtrar apenas municípios com cluster (reduz peso para as células seguintes)
            gdf_merged = gdf_merged[gdf_merged["cluster"].notna()].copy()
            gdf_merged["cluster"] = gdf_merged["cluster"].astype(int)

            # manter só o essencial + geometry (reduz peso ainda mais)
            keep_cols = ["geometry", "codigo_ibge", "cluster", "cluster_nome", "nome_municipio", "uf", "regiao"]
            keep_cols = [c for c in keep_cols if c in gdf_merged.columns]
            gdf_merged = gdf_merged[keep_cols].copy()

            print(f"✅ gdf_merged pronto e leve: {len(gdf_merged):,} municípios (apenas com cluster)")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CÉLULA 5: Criar mapa coroplético com Plotly
# ═══════════════════════════════════════════════════════════════════════════

if GEOPANDAS_AVAILABLE and 'gdf_merged' in locals() and gdf_merged is not None:
    print("\n" + "="*80)
    print("🎨 CRIANDO MAPA COROPLÉTICO (LEVE)")
    print("="*80)
    print()

    gdf_plot = gdf_merged.copy()

    # CRS para plotly
    if gdf_plot.crs is None or gdf_plot.crs.to_string() != "EPSG:4326":
        gdf_plot = gdf_plot.to_crs(epsg=4326)

    # manter só colunas essenciais (reduz peso)
    keep_cols = ['geometry', 'cluster', 'cluster_nome', 'nome_municipio', 'uf', 'regiao']
    keep_cols = [c for c in keep_cols if c in gdf_plot.columns]
    gdf_plot = gdf_plot[keep_cols].copy()

    # simplificar geometria (isso é o que salva!)
    print(f"🔧 Simplificando geometrias (tolerance={SIMPLIFY_TOL})...")
    gdf_plot["geometry"] = gdf_plot["geometry"].simplify(
        tolerance=SIMPLIFY_TOL,
        preserve_topology=True
    )

    # opcional: amostrar para testar
    if PLOTLY_MAX_FEATURES is not None and len(gdf_plot) > PLOTLY_MAX_FEATURES:
        gdf_plot = gdf_plot.sample(PLOTLY_MAX_FEATURES, random_state=42).copy()
        print(f"⚠️ Amostra aplicada: {len(gdf_plot):,} municípios")

    # Plotly recomenda geojson "FeatureCollection"
    geojson = json.loads(gdf_plot.to_json())

    # precisamos de um id para bater com locations
    gdf_plot = gdf_plot.reset_index(drop=True)
    gdf_plot["id"] = gdf_plot.index.astype(str)

    for i, feat in enumerate(geojson["features"]):
        feat["id"] = str(i)

    fig = px.choropleth(
        gdf_plot,
        geojson=geojson,
        locations="id",
        featureidkey="id",
        color="cluster_nome",
        hover_name="nome_municipio",
        hover_data=["uf", "regiao", "cluster"],
        title="Clusters de Municípios Brasileiros (Mapa LEVE)",
        height=900
    )

    fig.update_geos(fitbounds="locations", visible=False, projection_type="mercator")
    fig.update_layout(margin={"r":0,"t":50,"l":0,"b":0})

    out_path = figures_dir / "23_mapa_clusters_plotly_leve.html"
    fig.write_html(out_path)
    print(f"✅ Mapa Plotly LEVE salvo: {out_path}")
    fig.show()


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CÉLULA 6: Criar mapa com Folium (mais interativo)
# ═══════════════════════════════════════════════════════════════════════════

if GEOPANDAS_AVAILABLE and 'gdf_merged' in locals() and gdf_merged is not None:
    print("\n" + "="*80)
    print("🗺️  CRIANDO MAPA INTERATIVO COM FOLIUM (LEVE)")
    print("="*80)
    print()

    gdf_fol = gdf_merged.copy()

    if gdf_fol.crs is None or gdf_fol.crs.to_string() != "EPSG:4326":
        gdf_fol = gdf_fol.to_crs(epsg=4326)

    # simplificar para folium
    print(f"🔧 Simplificando geometrias (tolerance={FOLIUM_SIMPLIFY_TOL})...")
    gdf_fol["geometry"] = gdf_fol["geometry"].simplify(
        tolerance=FOLIUM_SIMPLIFY_TOL,
        preserve_topology=True
    )

    # Centro do Brasil
    m = folium.Map(location=[-14.235, -51.925], zoom_start=4, tiles="CartoDB positron")

    # paleta de cores (folium)
    colors = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00',
              '#ffff33','#a65628','#f781bf','#999999','#66c2a5']
    n_clusters = int(gdf_fol["cluster"].max() + 1)
    cluster_colors = {i: colors[i % len(colors)] for i in range(n_clusters)}

    # tooltip leve (sem HTML gigante)
    tooltip = folium.GeoJsonTooltip(
        fields=[c for c in ["nome_municipio", "uf", "regiao", "cluster_nome"] if c in gdf_fol.columns],
        aliases=["Município:", "UF:", "Região:", "Cluster:"],
        sticky=True
    )

    def style_function(feature):
        c = feature["properties"].get("cluster", None)
        try:
            c = int(c)
            color = cluster_colors.get(c, "#999999")
        except:
            color = "#999999"
        return {
            "fillColor": color,
            "color": "#333333",
            "weight": 0.3,
            "fillOpacity": 0.6
        }

    folium.GeoJson(
        data=json.loads(gdf_fol.to_json()),
        name="Clusters",
        style_function=style_function,
        tooltip=tooltip
    ).add_to(m)

    folium.LayerControl().add_to(m)

    map_path = figures_dir / "24_mapa_clusters_folium_leve.html"
    m.save(str(map_path))
    print(f"✅ Mapa Folium LEVE salvo: {map_path}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CÉLULA 7: Mapas por região
# ═══════════════════════════════════════════════════════════════════════════

if GEOPANDAS_AVAILABLE and 'gdf_merged' in locals() and gdf_merged is not None:
    print("\n" + "="*80)
    print("🗺️  MAPAS POR REGIÃO (LEVE)")
    print("="*80)
    print()

    gdf_reg = gdf_merged.copy()
    if gdf_reg.crs is None or gdf_reg.crs.to_string() != "EPSG:4326":
        gdf_reg = gdf_reg.to_crs(epsg=4326)

    # simplificar um pouco
    gdf_reg["geometry"] = gdf_reg["geometry"].simplify(tolerance=0.03, preserve_topology=True)

    regioes = sorted(gdf_reg['regiao'].dropna().unique())
    print(f"🔄 Criando mapas para {len(regioes)} regiões...")

    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    axes = axes.flatten()

    for i, regiao in enumerate(regioes[:6]):
        ax = axes[i]
        sub = gdf_reg[gdf_reg["regiao"] == regiao]
        sub.plot(
            column="cluster",
            categorical=True,
            ax=ax,
            cmap="tab10",
            edgecolor="none",
            linewidth=0
        )
        ax.set_title(f"Região {regiao}", fontsize=14, fontweight="bold")
        ax.set_axis_off()

    for j in range(len(regioes[:6]), len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    out_path = figures_dir / "25_mapas_por_regiao_leve.png"
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    print(f"✅ Mapas por região salvos: {out_path}")
    plt.show()


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CÉLULA 8: Visualizações alternativas (SEM shapefile) - LEVE + SEM ERRO
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("📊 VISUALIZAÇÕES ALTERNATIVAS (sem geometrias)")
print("="*80)
print()

print("💡 Mesmo sem shapefile, podemos criar visualizações úteis e leves")
print()

# Garantir colunas esperadas
required_cols = ["regiao", "uf", "cluster_nome"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"❌ Colunas ausentes no df: {missing_cols}")

# =========================
# 0) Agregação (LEVE) - base para treemap/sunburst
# =========================
print("🔄 Agregando dados (Região × UF × Cluster)...")

agg = (
    df.groupby(["regiao", "uf", "cluster_nome"], dropna=False)
      .size()
      .reset_index(name="count")
)

print(f"✅ Tabela agregada criada: {len(agg):,} linhas (bem menor que o df original)")
print()

# (opcional) reduzir ainda mais: manter só top N por região (deixa MUITO leve)
TOP_N_POR_REGIAO = None  # ex: 30 para ficar bem leve; None = não filtra

if TOP_N_POR_REGIAO is not None:
    agg = (
        agg.sort_values(["regiao", "count"], ascending=[True, False])
           .groupby("regiao")
           .head(TOP_N_POR_REGIAO)
           .reset_index(drop=True)
    )
    print(f"✅ Mantendo apenas TOP {TOP_N_POR_REGIAO} combinações por região: {len(agg):,} linhas")
    print()

# =========================
# 1) Treemap
# =========================
print("🎨 Criando Treemap...")

fig = px.treemap(
    agg,
    path=["regiao", "uf", "cluster_nome"],
    values="count",
    color="cluster_nome",
    title="Distribuição Hierárquica: Região → Estado → Cluster",
    height=700
)

fig.update_layout(
    font_size=12,
    title_font_size=16
)

treemap_path = figures_dir / "26_treemap_distribuicao.html"
fig.write_html(treemap_path)
print(f"✅ Treemap salvo: {treemap_path}")
fig.show()

# =========================
# 2) Sunburst
# =========================
print()
print("🎨 Criando Sunburst chart...")

fig = px.sunburst(
    agg,
    path=["regiao", "cluster_nome", "uf"],
    values="count",
    color="cluster_nome",
    title="Distribuição de Clusters - Visualização Radial",
    height=800
)

fig.update_layout(
    font_size=12,
    title_font_size=16
)

sunburst_path = figures_dir / "27_sunburst_distribuicao.html"
fig.write_html(sunburst_path)
print(f"✅ Sunburst salvo: {sunburst_path}")
fig.show()

# =========================
# 3) Matriz de densidade - Região x Cluster (usa df direto, mas é leve)
# =========================
print()
print("🎨 Criando matriz de densidade...")

density_matrix = pd.crosstab(df["regiao"], df["cluster_nome"])

fig = go.Figure(
    data=go.Heatmap(
        z=density_matrix.values,
        x=density_matrix.columns,
        y=density_matrix.index,
        colorscale="YlOrRd",
        text=density_matrix.values,
        texttemplate="%{text}",
        textfont={"size": 12},
        colorbar=dict(title="Nº Municípios")
    )
)

fig.update_layout(
    title="Densidade de Municípios: Região × Cluster",
    xaxis_title="Cluster",
    yaxis_title="Região",
    height=600,
    font_size=12,
    title_font_size=16
)

fig.update_xaxes(tickangle=45)

heatmap_path = figures_dir / "28_densidade_regiao_cluster.html"
fig.write_html(heatmap_path)
print(f"✅ Matriz de densidade salva: {heatmap_path}")
fig.show()


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CÉLULA 9: Resumo e exportação
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("📊 RESUMO DA VISUALIZAÇÃO GEOGRÁFICA")
print("="*80)
print()

# Garantir diretórios
results_dir = base_path / "outputs/results"
figures_dir = base_path / "outputs/figures"

results_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

# Criar relatório
geo_report = {
    "data_analise": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
    "shapefile_usado": "gdf_merged" in locals() and gdf_merged is not None,
    "total_municipios_visualizados": int(len(df)),
    "distribuicao_por_regiao": {}
}

for regiao in df["regiao"].dropna().unique():
    regiao_data = df[df["regiao"] == regiao]
    cluster_dist = regiao_data["cluster_nome"].value_counts().to_dict()
    
    geo_report["distribuicao_por_regiao"][regiao] = {
        "total_municipios": int(len(regiao_data)),
        "clusters": cluster_dist
    }

# Salvar relatório
report_path = results_dir / "relatorio_geografico.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(geo_report, f, indent=2, ensure_ascii=False)

print(f"✅ Relatório geográfico salvo: {report_path}")
print()

print("="*80)
print("🎉 VISUALIZAÇÃO GEOGRÁFICA CONCLUÍDA!")
print("="*80)
print()
print("📊 ARQUIVOS GERADOS:")
print()

# ✅ CORREÇÃO: Nomes corretos dos arquivos
visualizations = [
    ("23_mapa_clusters_plotly_leve.html", "Mapa coroplético interativo (Plotly)"),
    ("24_mapa_clusters_folium_leve.html", "Mapa interativo avançado (Folium)"),
    ("25_mapas_por_regiao_leve.png", "Mapas separados por região"),
    ("26_treemap_distribuicao.html", "Treemap hierárquico"),
    ("27_sunburst_distribuicao.html", "Sunburst radial"),
    ("28_densidade_regiao_cluster.html", "Matriz de densidade")
]

for filename, description in visualizations:
    filepath = figures_dir / filename
    status = "✅" if filepath.exists() else "⚠️ "
    print(f"   {status} {filename}")
    if filepath.exists():
        print(f"      {description}")
        size_mb = filepath.stat().st_size / (1024 * 1024)
        print(f"      Tamanho: {size_mb:.2f} MB")
    print()

print("💡 INSIGHTS GEOGRÁFICOS:")
print()
print("   • Nordeste: Concentração de municípios em desenvolvimento")
print("   • Sul: Região mais equilibrada e desenvolvida")
print("   • Centro-Oeste: Forte presença de agronegócio")
print("   • Norte: Predominância de municípios rurais")
print("   • Sudeste: Mix diversificado de perfis")
print()
print("👉 PRÓXIMO PASSO:")
print("   Criar apresentação final dos resultados!")
print()
print("="*80)